# Auditoría final desde el ledger público

Este notebook recalcula las cifras finales de la memoria usando solo el ledger anonimizado publicado en `results/final_candidate_actions_anonymized.csv`.

No necesita el corpus privado ni los modelos entrenados. Por eso es el notebook público más directo para auditar el resultado final.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

ledger_path = ROOT / "results" / "final_candidate_actions_anonymized.csv"
summary_path = ROOT / "results" / "final_candidate_summary.csv"

ledger = pd.read_csv(ledger_path)
published = pd.read_csv(summary_path)
ledger.head()

,action_id,session_day,market_hash,token_hash,seconds_from_window_start,volatility_regime,net_cost_0p25,net_cost_0p5,net_cost_1p0
0,1,2026-06-06,2f2ff968d9c433db,e6c8c93626f1aaa2,-51.767,high,-0.676037,-0.852074,-1.204147
1,2,2026-06-06,2f2ff968d9c433db,e6c8c93626f1aaa2,-49.767,high,-1.676036,-1.852073,-2.204146
2,3,2026-06-06,2f2ff968d9c433db,e6c8c93626f1aaa2,-47.767,high,0.323962,0.147925,-0.204148
3,4,2026-06-06,6f22660e9f1ea2fa,970332538f9361a6,-46.400,high,-7.673239,-7.846478,-8.192957
4,5,2026-06-06,336b363f49423d6f,06344403bd597d56,-52.400,high,-7.677332,-7.854664,-8.209328


## Resultado OOS limpio

El resultado confirmatorio es el especialista base sin el filtro de volatilidad posterior.

In [2]:
base = ledger
base_table = pd.Series({
    "n_actions": len(base),
    "mean_net_0p25": base["net_cost_0p25"].mean(),
    "mean_net_0p5": base["net_cost_0p5"].mean(),
    "mean_net_1p0": base["net_cost_1p0"].mean(),
    "positive_days_0p5": (base.groupby("session_day")["net_cost_0p5"].mean() > 0).sum(),
    "n_days": base["session_day"].nunique(),
})
base_table

n_actions            754.000000
mean_net_0p25          0.536987
mean_net_0p5           0.349173
mean_net_1p0          -0.026456
positive_days_0p5      3.000000
n_days                 5.000000
dtype: float64

## Diagnóstico post-hoc de volatilidad

La separación por baja volatilidad es diagnóstica: se observó después de inspeccionar el bloque fresh y por tanto no debe leerse como una segunda validación OOS.

In [3]:
regime_table = (
    ledger.groupby("volatility_regime")
    .agg(
        n=("action_id", "size"),
        net_0p25=("net_cost_0p25", "mean"),
        net_0p5=("net_cost_0p5", "mean"),
        net_1p0=("net_cost_1p0", "mean"),
        positive_days=("net_cost_0p5", lambda x: (ledger.loc[x.index].groupby("session_day")["net_cost_0p5"].mean() > 0).sum()),
    )
    .sort_index()
)
regime_table

,n,net_0p25,net_0p5,net_1p0,positive_days
volatility_regime,,,,,
high,435,0.003656,-0.183493,-0.557791,1
low,318,1.257771,1.069000,0.691460,5
missing,1,3.326763,3.153524,2.807045,1


## Incertidumbre agrupada por mercado

El bootstrap agrupado evita tratar como independientes varias señales solapadas del mismo mercado.

In [4]:
BOOTSTRAP_SEED = 42
N_BOOTSTRAP = 5_000

low = ledger[ledger["volatility_regime"] == "low"].copy()
grouped = low.groupby("market_hash", sort=False)["net_cost_0p5"]
sums = grouped.sum().to_numpy(dtype=float)
counts = grouped.size().to_numpy(dtype=float)

rng = np.random.default_rng(BOOTSTRAP_SEED)
means = np.empty(N_BOOTSTRAP)
for i in range(N_BOOTSTRAP):
    sampled = rng.integers(0, len(sums), size=len(sums))
    means[i] = sums[sampled].sum() / counts[sampled].sum()

cluster_ci90 = np.quantile(means, [0.05, 0.95])
cluster_p_positive = (means > 0).mean()

pd.Series({
    "low_vol_actions": len(low),
    "unique_markets": low["market_hash"].nunique(),
    "max_actions_per_market": low.groupby("market_hash").size().max(),
    "market_cluster_ci90_low": cluster_ci90[0],
    "market_cluster_ci90_high": cluster_ci90[1],
    "market_cluster_p_positive": cluster_p_positive,
})

low_vol_actions              318.000000
unique_markets               156.000000
max_actions_per_market         8.000000
market_cluster_ci90_low        0.192680
market_cluster_ci90_high       2.003563
market_cluster_p_positive      0.976000
dtype: float64

## Comparación con el resumen publicado

El script equivalente, apto para CI o consola, es:

```bash
python scripts/experiments/recompute_final_summary_from_public_ledger.py --check
```

In [5]:
published.head(10)

,scope,metric,value,unit
0,base_oos,n_actions,754.000000,actions
1,base_oos,mean_net_cost_0p25,0.536987,ticks/action
2,base_oos,mean_net_cost_0p5,0.349173,ticks/action
3,base_oos,mean_net_cost_1p0,-0.026456,ticks/action
4,low_vol_posthoc,n_actions,318.000000,actions
5,low_vol_posthoc,mean_net_cost_0p25,1.257770,ticks/action
6,low_vol_posthoc,mean_net_cost_0p5,1.069000,ticks/action
7,low_vol_posthoc,mean_net_cost_1p0,0.691460,ticks/action
8,high_vol_posthoc,n_actions,435.000000,actions
9,high_vol_posthoc,mean_net_cost_0p25,0.003656,ticks/action
